In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.layers import GRU
from tensorflow.keras.optimizers import Adam
# Load the datasets
data_train = pd.read_csv('/content/sarcasm_mal_train.csv')
data_dev = pd.read_csv('/content/sarcasm_mal_dev.csv')
data_test = pd.read_csv('/content/sarcasm_mal_test_without_labels.csv')  # Assume this is your test dataset

# Combine the training and development sets
texts_combined = data_train['Text'].astype(str).values.tolist() + data_dev['Text'].astype(str).values.tolist()
labels_combined = np.concatenate([data_train['labels'].values, data_dev['labels'].values])

# Extract test texts
texts_test = data_test['Text'].astype(str).values.tolist()

# Tokenization parameters
max_words = 10000  # Maximum number of words to keep
max_len = 100      # Maximum sequence length

# Tokenize the text
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(texts_combined)  # Fit on combined texts
sequences_combined = tokenizer.texts_to_sequences(texts_combined)
sequences_test = tokenizer.texts_to_sequences(texts_test)
padded_sequences_combined = pad_sequences(sequences_combined, maxlen=max_len, padding='post')
padded_sequences_test = pad_sequences(sequences_test, maxlen=max_len, padding='post')

# Encode labels
label_encoder = LabelEncoder()
labels_encoded_combined = label_encoder.fit_transform(labels_combined)

# Convert labels to numpy arrays and ensure they are integers
labels_encoded_combined = np.array(labels_encoded_combined).astype(np.int32)

model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=256, input_length=max_len))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(32)))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))  # Add dense layer after LSTM to reduce dimensions
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))  # Output layer

# Compile the model
optimizer = Adam(learning_rate=0.0001)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Print model summary
model.summary()

# Train the model on the combined training set
model.fit(
    padded_sequences_combined,
    labels_encoded_combined,
    epochs=10,
    batch_size=32,
    validation_split=0.1  # Reserve 10% for validation during training
)

# Predict the labels for the test set
y_pred_test = (model.predict(padded_sequences_test) > 0.5).astype("int32")

# Create a DataFrame with predictions
predictions = pd.DataFrame({
    'Text': texts_test,
    'Predicted_Label': y_pred_test.flatten()
})

# Save the predictions to a CSV file
predictions.to_csv('/content/Codespark_Malayalam_run2.csv', index=False)

print("Predictions saved to Codespark_Malayalam_run2.csv")

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_11 (Bidirectional)     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_8 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_12 (Bidirectional)     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_9 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_13 (Bidirectional)     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 378s 740ms/step - accuracy: 0.7904 - loss: 0.5464 - val_accuracy: 0.8121 - val_loss: 0.4704
Epoch 2/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 383s 742ms/step - accuracy: 0.8137 - loss: 0.4741 - val_accuracy: 0.8390 - val_loss: 0.3789
Epoch 3/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 378s 735ms/step - accuracy: 0.8732 - loss: 0.3355 - val_accuracy: 0.8502 - val_loss: 0.3801
Epoch 4/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 388s 749ms/step - accuracy: 0.9041 - loss: 0.2650 - val_accuracy: 0.8527 - val_loss: 0.4146
Epoch 5/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 383s 751ms/step - accuracy: 0.9303 - loss: 0.2171 - val_accuracy: 0.8290 - val_loss: 0.4337
Epoch 6/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 376s 738ms/step - accuracy: 0.9451 - loss: 0.1846 - val_accuracy: 0.8290 - val_loss: 0.4956
Epoch 7/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 349s 774ms/step - accuracy: 0.9523 - loss: 0.1563 - val_accuracy: 0.8340 - val_loss: 0.5183
Epoch 8/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 373s 754ms/step - accuracy: 0.9609 -

#for txt values

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam

# Load the datasets
data_train = pd.read_csv('/content/sarcasm_mal_train.csv')
data_dev = pd.read_csv('/content/sarcasm_mal_dev.csv')
data_test = pd.read_csv('/content/sarcasm_mal_test_without_labels.csv')  # Assume this is your test dataset

# Combine the training and development sets
texts_combined = data_train['Text'].astype(str).values.tolist() + data_dev['Text'].astype(str).values.tolist()
labels_combined = np.concatenate([data_train['labels'].values, data_dev['labels'].values])

# Extract test texts
texts_test = data_test['Text'].astype(str).values.tolist()

# Tokenization parameters
max_words = 10000  # Maximum number of words to keep
max_len = 100      # Maximum sequence length

# Tokenize the text
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(texts_combined)  # Fit on combined texts
sequences_combined = tokenizer.texts_to_sequences(texts_combined)
sequences_test = tokenizer.texts_to_sequences(texts_test)
padded_sequences_combined = pad_sequences(sequences_combined, maxlen=max_len, padding='post')
padded_sequences_test = pad_sequences(sequences_test, maxlen=max_len, padding='post')

# Encode labels
label_encoder = LabelEncoder()
labels_encoded_combined = label_encoder.fit_transform(labels_combined)

# Convert labels to numpy arrays and ensure they are integers
labels_encoded_combined = np.array(labels_encoded_combined).astype(np.int32)

# Build the model
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=256, input_length=max_len))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(32)))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))  # Add dense layer after LSTM to reduce dimensions
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))  # Output layer

# Compile the model
optimizer = Adam(learning_rate=0.0001)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Print model summary
model.summary()

# Train the model on the combined training set
model.fit(
    padded_sequences_combined,
    labels_encoded_combined,
    epochs=10,
    batch_size=32,
    validation_split=0.1  # Reserve 10% for validation during training
)

# Predict the labels for the test set
y_pred_test = (model.predict(padded_sequences_test) > 0.5).astype("int32")

# Map 0/1 predictions to "Non-sarcastic"/"Sarcastic"
label_mapping = {0: "Non-sarcastic", 1: "Sarcastic"}
predicted_labels = [label_mapping[label] for label in y_pred_test.flatten()]

# Create a DataFrame with predictions
predictions = pd.DataFrame({
    'Text': texts_test,
    'Predicted_Label': predicted_labels
})

# Save the predictions to a CSV file
predictions.to_csv('/content/Codespark_Malayalam_run2(1).csv', index=False)

print("Predictions saved to Codespark_Malayalam_run2.csv")


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_17 (Bidirectional)     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_16 (Dropout)                 │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_18 (Bidirectional)     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_17 (Dropout)                 │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_19 (Bidirectional)     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_18 (Dropout)                 │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_19 (Dropout)                 │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 348s 745ms/step - accuracy: 0.8004 - loss: 0.5367 - val_accuracy: 0.8115 - val_loss: 0.4700
Epoch 2/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 384s 751ms/step - accuracy: 0.8196 - loss: 0.4674 - val_accuracy: 0.8439 - val_loss: 0.3768
Epoch 3/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 380s 746ms/step - accuracy: 0.8729 - loss: 0.3353 - val_accuracy: 0.8508 - val_loss: 0.3701
Epoch 4/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 395s 775ms/step - accuracy: 0.9058 - loss: 0.2622 - val_accuracy: 0.8477 - val_loss: 0.4092
Epoch 5/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 372s 754ms/step - accuracy: 0.9271 - loss: 0.2196 - val_accuracy: 0.8464 - val_loss: 0.4353
Epoch 6/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 336s 745ms/step - accuracy: 0.9432 - loss: 0.1817 - val_accuracy: 0.8340 - val_loss: 0.4699
Epoch 7/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 387s 758ms/step - accuracy: 0.9087 - loss: 0.2341 - val_accuracy: 0.8327 - val_loss: 0.5006
Epoch 8/10
451/451 ━━━━━━━━━━━━━━━━━━━━ 344s 763ms/step - accuracy: 0.9490 -